In [0]:
-- 1. Check the table structure and column names
DESCRIBE TABLE pondolens.dev_bronze.hospital;

-- 2. Dataset overview: row count, time/geographic coverage, and numeric range
SELECT
  COUNT(*) AS row_count,
  MIN(year) AS first_year,
  MAX(year) AS last_year,
  COUNT(DISTINCT year) AS distinct_years,
  COUNT(DISTINCT region_name) AS distinct_regions,
  COUNT(DISTINCT hospital_type) AS distinct_hospital_types,
  MIN(TRY_CAST(regulated_hospital_count AS BIGINT)) AS min_hospital_count,
  MAX(TRY_CAST(regulated_hospital_count AS BIGINT)) AS max_hospital_count,
  AVG(TRY_CAST(regulated_hospital_count AS DOUBLE)) AS avg_hospital_count
FROM pondolens.dev_bronze.hospital;

-- 3. Completeness: missing values
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN year IS NULL THEN 1 ELSE 0 END) AS missing_year,
  SUM(CASE WHEN region_name IS NULL OR TRIM(region_name) = '' THEN 1 ELSE 0 END) AS missing_region,
  SUM(CASE WHEN hospital_type IS NULL OR TRIM(hospital_type) = '' THEN 1 ELSE 0 END) AS missing_hospital_type,
  SUM(CASE WHEN regulated_hospital_count IS NULL THEN 1 ELSE 0 END) AS missing_hospital_count,
  SUM(CASE WHEN unit IS NULL OR TRIM(unit) = '' THEN 1 ELSE 0 END) AS missing_unit
FROM pondolens.dev_bronze.hospital;

-- 4. Coverage by year: expected 17 historical source regions
SELECT
  year,
  COUNT(DISTINCT region_name) AS region_count,
  COUNT(DISTINCT hospital_type) AS hospital_type_count,
  COUNT(*) AS row_count,
  COLLECT_SET(region_name) AS regions_found
FROM pondolens.dev_bronze.hospital
GROUP BY year
ORDER BY year;

-- 5. Check expected hospital categories by year and region
-- Expected categories: Total, Government, Private
SELECT
  year,
  region_name,
  COUNT(DISTINCT hospital_type) AS category_count,
  COLLECT_SET(hospital_type) AS categories_found
FROM pondolens.dev_bronze.hospital
GROUP BY year, region_name
HAVING COUNT(DISTINCT hospital_type) <> 3
ORDER BY year, region_name;

-- 6. Show possible values for categories and units
SELECT 'hospital_type' AS column_name, hospital_type AS value, COUNT(*) AS row_count
FROM pondolens.dev_bronze.hospital
GROUP BY hospital_type

UNION ALL

SELECT 'unit' AS column_name, unit AS value, COUNT(*) AS row_count
FROM pondolens.dev_bronze.hospital
GROUP BY unit
ORDER BY column_name, value;

-- 7. Uniqueness: duplicate business keys
-- Expected grain/key: one row per year + region + hospital type
SELECT
  year,
  region_name,
  hospital_type,
  COUNT(*) AS duplicate_count
FROM pondolens.dev_bronze.hospital
GROUP BY year, region_name, hospital_type
HAVING COUNT(*) > 1
ORDER BY year, region_name, hospital_type;

-- 8. Validity: years, counts, categories, and units
SELECT
  SUM(CASE WHEN year < 2021 OR year > 2024 THEN 1 ELSE 0 END) AS out_of_scope_year,
  SUM(CASE WHEN TRY_CAST(regulated_hospital_count AS BIGINT) IS NULL THEN 1 ELSE 0 END) AS non_numeric_count,
  SUM(CASE WHEN TRY_CAST(regulated_hospital_count AS BIGINT) < 0 THEN 1 ELSE 0 END) AS negative_count,
  SUM(CASE WHEN hospital_type NOT IN ('Total', 'Government', 'Private') THEN 1 ELSE 0 END) AS unexpected_hospital_type,
  SUM(CASE WHEN LOWER(TRIM(unit)) <> 'hospitals' THEN 1 ELSE 0 END) AS unexpected_unit
FROM pondolens.dev_bronze.hospital;

-- 9. Consistency: Total should equal Government + Private
WITH hospital_counts AS (
  SELECT
    year,
    region_name,
    MAX(CASE WHEN hospital_type = 'Total'
      THEN TRY_CAST(regulated_hospital_count AS BIGINT) END) AS total_count,
    MAX(CASE WHEN hospital_type = 'Government'
      THEN TRY_CAST(regulated_hospital_count AS BIGINT) END) AS government_count,
    MAX(CASE WHEN hospital_type = 'Private'
      THEN TRY_CAST(regulated_hospital_count AS BIGINT) END) AS private_count
  FROM pondolens.dev_bronze.hospital
  GROUP BY year, region_name
)
SELECT
  *,
  total_count - government_count - private_count AS difference
FROM hospital_counts
WHERE total_count <> government_count + private_count
   OR total_count IS NULL
   OR government_count IS NULL
   OR private_count IS NULL
ORDER BY year, region_name;

-- 10. Regional sums for comparison with national source totals
SELECT
  year,
  hospital_type,
  SUM(TRY_CAST(regulated_hospital_count AS BIGINT)) AS regional_sum
FROM pondolens.dev_bronze.hospital
GROUP BY year, hospital_type
ORDER BY year, hospital_type;
